# Gold 2K Evaluation — All Models

**Purpose:** Evaluation of BiLSTM, mBERT, and Ensemble on the **human-annotated** gold 2K set from the OSCAR corpus.


**Evaluation set:**
- 2,008 sentences from `gold_2k_annotation_Main.xlsx` (Column B: original, Column C: correction)
- Metadata from `gold_2k_metadata_final.json` (position, rules, complexity)
- Error analysis broken down by position type, complexity, and participle type

## Step 1: Setup

In [ ]:
import os
import re
import json
import math
import random
import difflib
import time
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import classification_report, f1_score, precision_recall_fscore_support

In [4]:
# Paths 
# Adjust them to match your local directory structure
# See README.md for expected folder layout.

# DATA_DIR       = Path("./Data Full/results_clean")
# GOLD_XLSX      = Path("./Data Full/gold_2k_annotation_Main.xlsx")
# GOLD_META      = Path("./Data Full/gold_2k_metadata_final.json")
# BILSTM_VOCAB   = Path("./Weights - train_bilstm_v3/armenian_vocab.json")
# BILSTM_CKPT    = Path("./Weights - train_bilstm_v3/bilstm_v3_best.pt")
# MBERT_WEIGHTS  = Path("./mBert/mbert_best.pt")

MBERT_NAME     = "bert-base-multilingual-cased"

LABEL_MAP  = {"O": 0, "COMMA_AFTER": 1, "BUTH_AFTER": 2, "REMOVE_COMMA": 3}
LABEL_LIST = ["O", "COMMA_AFTER", "BUTH_AFTER", "REMOVE_COMMA"]
NUM_LABELS = 4

ENSEMBLE_ALPHA = 0.45  # from ensemble optimization

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cpu")

for p in [GOLD_XLSX, BILSTM_VOCAB, BILSTM_CKPT, MBERT_WEIGHTS]:
    assert p.exists(), f"Missing: {p}"
print("All paths verified ✓")

All paths verified ✓


## Step 2: Label Pipeline (shared)

In [7]:
ARMENIAN_PUNCT = '\u055b\u055c\u055d\u055e\u055f\u0589'
ARMENIAN_PUNCT_SET = set(ARMENIAN_PUNCT)

def tokenize_armenian(text):
    if not isinstance(text, str) or not text.strip():
        return []
    for ch in ARMENIAN_PUNCT:
        text = text.replace(ch, f' {ch} ')
    return re.findall(r'[\w\u0530-\u058F]+|[^\w\s]', text)

def generate_labels(orig_tokens, corr_tokens):
    labels = [0] * len(orig_tokens)
    sm = difflib.SequenceMatcher(None, orig_tokens, corr_tokens)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == 'equal': continue
        if tag in ('delete', 'replace'):
            for i in range(i1, i2):
                if orig_tokens[i] == ',':
                    labels[i] = 3
        if tag in ('insert', 'replace'):
            for j in range(j1, j2):
                tok = corr_tokens[j]
                if tok == ',' and i1 > 0:
                    labels[i1 - 1] = 1
                elif tok == '\u055d' and i1 > 0:
                    labels[i1 - 1] = 2
    return labels

def is_punct_token(tok):
    if len(tok) == 1 and tok in ARMENIAN_PUNCT_SET:
        return True
    return not re.match(r'[\w\u0530-\u058F]', tok)

def extract_word_labels(orig_tokens, label_ids):
    words, word_labels = [], []
    for tok, lbl in zip(orig_tokens, label_ids):
        if is_punct_token(tok):
            if lbl == 3 and word_labels and word_labels[-1] == 0:
                word_labels[-1] = 3
            continue
        words.append(tok)
        word_labels.append(lbl)
    return words, word_labels

print("Label pipeline ready ✓")

Label pipeline ready ✓


## Step 3: Load Gold 2K

Reads the xlsx (Column B = original, Column C = human correction).
Skips rows where the correction is empty (not yet annotated).

In [10]:
import openpyxl

wb = openpyxl.load_workbook(GOLD_XLSX, read_only=True)
ws = wb.active

gold_samples = []
skipped_empty = 0
skipped_error = 0

for row_idx, row in enumerate(ws.iter_rows(min_row=2, values_only=True), start=2):
    if len(row) < 3:
        skipped_error += 1
        continue

    row_num, original, corrected = row[0], row[1], row[2]

    # Skip unannotated rows
    if not corrected or not isinstance(corrected, str) or not corrected.strip():
        skipped_empty += 1
        continue
    if not original or not isinstance(original, str) or not original.strip():
        skipped_error += 1
        continue

    # If correction is identical to original → all O (no changes needed)
    orig_tokens = tokenize_armenian(original)
    corr_tokens = tokenize_armenian(corrected)
    if not orig_tokens:
        skipped_error += 1
        continue

    if not corr_tokens:
        corr_tokens = orig_tokens  # treat as no correction

    labels = generate_labels(orig_tokens, corr_tokens)
    words, word_labels = extract_word_labels(orig_tokens, labels)
    if not words:
        skipped_error += 1
        continue

    gold_samples.append({
        "row": row_num,
        "original": original,
        "corrected": corrected,
        "words": words,
        "labels": word_labels,
    })

wb.close()

print(f"Gold 2K loaded: {len(gold_samples)} annotated sentences")
print(f"  Skipped (empty correction): {skipped_empty}")
print(f"  Skipped (errors): {skipped_error}")

if len(gold_samples) < 100:
    print("\n⚠ WARNING: Very few annotated sentences. Results may not be reliable.")
    print("  Annotate more rows in Column C of the xlsx before re-running.")

# Label distribution
gold_label_counts = Counter()
for s in gold_samples:
    gold_label_counts.update(s["labels"])
total_tokens = sum(gold_label_counts.values())
print(f"\nGold 2K label distribution ({total_tokens:,} tokens):")
for i, name in enumerate(LABEL_LIST):
    c = gold_label_counts.get(i, 0)
    print(f"  {name:15s}: {c:>7,} ({100*c/total_tokens:.2f}%)")

Gold 2K loaded: 2008 annotated sentences
  Skipped (empty correction): 0
  Skipped (errors): 0

Gold 2K label distribution (45,492 tokens):
  O              :  44,977 (98.87%)
  COMMA_AFTER    :     227 (0.50%)
  BUTH_AFTER     :     244 (0.54%)
  REMOVE_COMMA   :      44 (0.10%)


In [12]:
# ── Load metadata for error analysis ───────────────────────────
gold_meta = {}
if GOLD_META.exists():
    with open(GOLD_META, 'r', encoding='utf-8') as f:
        meta_list = json.load(f)
    # Index by sentence text for lookup
    for rec in meta_list:
        text = rec.get("text", "")
        if text:
            gold_meta[text] = rec
    print(f"Metadata loaded: {len(gold_meta)} entries")
    # Attach metadata to samples
    matched = 0
    for s in gold_samples:
        meta = gold_meta.get(s["original"])
        if meta:
            tags = meta.get("sampling_tags", {})
            s["position"] = tags.get("primary_position", "UNKNOWN")
            s["complexity"] = tags.get("complexity", "unknown")
            s["rules"] = tags.get("applicable_rules", [])
            s["participle_type"] = tags.get("participle_type", "unknown")
            matched += 1
        else:
            s["position"] = "UNKNOWN"
            s["complexity"] = "unknown"
            s["rules"] = []
            s["participle_type"] = "unknown"
    print(f"  Matched to samples: {matched}/{len(gold_samples)}")
else:
    print("Metadata file not found — error analysis by position/complexity unavailable")
    for s in gold_samples:
        s["position"] = "UNKNOWN"
        s["complexity"] = "unknown"
        s["rules"] = []
        s["participle_type"] = "unknown" 

Metadata loaded: 1990 entries
  Matched to samples: 1612/2008


## Step 4: Loading Models

In [15]:
# BiLSTM
class BiLSTMPunctuator(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size, num_layers,
                 dropout, num_classes, pretrained_emb=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        if pretrained_emb is not None:
            self.embedding.weight.data.copy_(pretrained_emb)
            self.embedding.weight.requires_grad = False
        self.lstm = nn.LSTM(input_size=emb_dim, hidden_size=hidden_size,
                           num_layers=num_layers, batch_first=True,
                           bidirectional=True,
                           dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        out, _ = self.lstm(self.embedding(x))
        return self.classifier(self.dropout(out))

with open(BILSTM_VOCAB, 'r', encoding='utf-8') as f:
    bilstm_vocab = json.load(f)

ckpt = torch.load(BILSTM_CKPT, map_location="cpu", weights_only=False)
bp = ckpt.get("best_params", ckpt.get("hyperparameters", {}))

bilstm_model = BiLSTMPunctuator(
    vocab_size=ckpt["vocab_size"], emb_dim=ckpt["embedding_dim"],
    hidden_size=bp["hidden_size"], num_layers=bp["num_layers"],
    dropout=bp["dropout"], num_classes=ckpt["num_classes"],
)
bilstm_model.load_state_dict(ckpt["model_state_dict"])
bilstm_model.eval()
print(f"BiLSTM loaded: hidden={bp['hidden_size']}, layers={bp['num_layers']}")

BiLSTM v3 loaded: hidden=128, layers=1


In [17]:
# ── mBERT ──────────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForTokenClassification
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

mbert_tokenizer = AutoTokenizer.from_pretrained(MBERT_NAME)
mbert_model = AutoModelForTokenClassification.from_pretrained(
    MBERT_NAME, num_labels=NUM_LABELS)
mbert_model.load_state_dict(torch.load(MBERT_WEIGHTS, map_location="cpu", weights_only=True))
mbert_model.eval()
print(f"mBERT loaded: {sum(p.numel() for p in mbert_model.parameters()):,} params")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

mBERT loaded: 177,265,924 params


## Step 5: Get Predictions from All Models

In [20]:
UNK_IDX = bilstm_vocab.get("<UNK>", 1)

def get_bilstm_probs(words):
    ids = [bilstm_vocab.get(w, UNK_IDX) for w in words[:256]]
    x = torch.tensor([ids], dtype=torch.long)
    with torch.no_grad():
        logits = bilstm_model(x)
    return F.softmax(logits[0], dim=-1).numpy()

def get_mbert_probs(words):
    encoding = mbert_tokenizer(
        words, is_split_into_words=True,
        max_length=128, truncation=True,
        padding=False, return_tensors="pt"
    )
    word_ids = encoding.word_ids(batch_index=0)
    with torch.no_grad():
        logits = mbert_model(
            input_ids=encoding["input_ids"],
            attention_mask=encoding["attention_mask"]
        ).logits[0]
    all_probs = F.softmax(logits, dim=-1).numpy()
    word_probs = []
    prev_wid = None
    for i, wid in enumerate(word_ids):
        if wid is None: continue
        if wid != prev_wid:
            word_probs.append(all_probs[i])
        prev_wid = wid
    return np.array(word_probs)

print("Prediction functions ready ✓")

Prediction functions ready ✓


In [22]:
# ── Run all models on gold 2K ──────────────────────────────────
print("Running inference on gold 2K...")
t0 = time.time()

for s in gold_samples:
    words = s["words"]
    labels = s["labels"]

    bp = get_bilstm_probs(words)
    mp = get_mbert_probs(words)

    n = min(len(labels), len(bp), len(mp))
    s["n_aligned"] = n
    s["labels_aligned"] = labels[:n]

    # BiLSTM predictions
    s["bilstm_preds"] = bp[:n].argmax(axis=1).tolist()

    # mBERT predictions
    s["mbert_preds"] = mp[:n].argmax(axis=1).tolist()

    # Ensemble predictions
    combined = ENSEMBLE_ALPHA * bp[:n] + (1 - ENSEMBLE_ALPHA) * mp[:n]
    s["ensemble_preds"] = combined.argmax(axis=1).tolist()

elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s ({len(gold_samples)} sentences)")

# Free model memory
del bilstm_model, mbert_model
import gc; gc.collect()

Running inference on gold 2K...
Done in 91.3s (2008 sentences)


99

## Step 6: Evaluation Results

In [24]:
def evaluate_model(gold_samples, pred_key, model_name):
    """Evaluate a model's predictions against gold labels."""
    all_preds, all_labels = [], []
    for s in gold_samples:
        all_preds.extend(s[pred_key])
        all_labels.extend(s["labels_aligned"])

    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    ins_f1 = f1_score(all_labels, all_preds, labels=[1, 2], average="macro", zero_division=0)

    preds_3c = [0 if x == 3 else x for x in all_preds]
    labels_3c = [0 if x == 3 else x for x in all_labels]
    f1_3c = f1_score(labels_3c, preds_3c, labels=[0,1,2], average="macro", zero_division=0)

    print(f"\n{'='*60}")
    print(f"{model_name} — Gold 2K Evaluation")
    print(f"{'='*60}")
    print(classification_report(all_labels, all_preds,
          target_names=LABEL_LIST, digits=3, zero_division=0))

    pc, lc = Counter(all_preds), Counter(all_labels)
    print("Distribution:")
    for i, name in enumerate(LABEL_LIST):
        print(f"  {name:15s}: {lc.get(i,0):>7,} true, {pc.get(i,0):>7,} pred")

    print(f"\nMacro-F1:          {macro_f1:.4f}")
    print(f"3-class remap F1:  {f1_3c:.4f}")
    print(f"Insertion-only F1: {ins_f1:.4f}")

    return {
        "macro_f1": round(macro_f1, 5),
        "f1_3c": round(f1_3c, 5),
        "ins_f1": round(ins_f1, 5),
        "all_preds": all_preds,
        "all_labels": all_labels,
    }

bilstm_res = evaluate_model(gold_samples, "bilstm_preds", "BiLSTM")
mbert_res = evaluate_model(gold_samples, "mbert_preds", "mBERT")
ensemble_res = evaluate_model(gold_samples, "ensemble_preds", f"Ensemble (α={ENSEMBLE_ALPHA})")


BiLSTM v3 — Gold 2K Evaluation
              precision    recall  f1-score   support

           O      0.996     0.970     0.983     44418
 COMMA_AFTER      0.256     0.617     0.362       227
  BUTH_AFTER      0.158     0.642     0.253       243
REMOVE_COMMA      0.033     0.091     0.048        44

    accuracy                          0.966     44932
   macro avg      0.361     0.580     0.412     44932
weighted avg      0.987     0.966     0.975     44932

Distribution:
  O              :  44,418 true,  43,275 pred
  COMMA_AFTER    :     227 true,     547 pred
  BUTH_AFTER     :     243 true,     989 pred
  REMOVE_COMMA   :      44 true,     121 pred

Macro-F1:          0.4116
3-class remap F1:  0.5332
Insertion-only F1: 0.3075

mBERT — Gold 2K Evaluation
              precision    recall  f1-score   support

           O      0.998     0.975     0.986     44418
 COMMA_AFTER      0.301     0.722     0.425       227
  BUTH_AFTER      0.209     0.815     0.332       243
REMOVE_COMM

## Step 7: Cross-Model Comparison Table

In [26]:
print("=" * 85)
print("CROSS-MODEL COMPARISON — Gold 2K (human-annotated)")
print("=" * 85)
print(f"{'Model':<35} {'Macro-F1':>10} {'Ins F1':>8} {'3c F1':>8} {'Val F1':>8}")
print("-" * 75)
print(f"{'BiLSTM':<35} {bilstm_res['macro_f1']:>10.4f} {bilstm_res['ins_f1']:>8.4f} "
      f"{bilstm_res['f1_3c']:>8.4f} {'0.4104':>8}")
print(f"{'mBERT':<35} {mbert_res['macro_f1']:>10.4f} {mbert_res['ins_f1']:>8.4f} "
      f"{mbert_res['f1_3c']:>8.4f} {'0.4854':>8}")
print(f"{'Ensemble (α=0.45)':<35} {ensemble_res['macro_f1']:>10.4f} {ensemble_res['ins_f1']:>8.4f} "
      f"{ensemble_res['f1_3c']:>8.4f} {'0.4919':>8}")
print()
print("Val F1 = performance on Gemini-labeled unfiltered val (for reference)")
print("Gold 2K = performance on human-annotated data (the real test)")

# Check if gold scores are higher than val scores
best_gold = ensemble_res['macro_f1']
if best_gold > 0.4919:
    print(f"\n>>> Gold 2K scores HIGHER than val — label noise ceiling confirmed!")
    print(f"    Models are better than Gemini labels suggested.")
elif best_gold < 0.4919 * 0.9:
    print(f"\n>>> Gold 2K scores LOWER — models may have overfit to Gemini-specific patterns.")
else:
    print(f"\n>>> Gold 2K scores comparable to val — consistent performance.")

CROSS-MODEL COMPARISON — Gold 2K (human-annotated)
Model                                 Macro-F1   Ins F1    3c F1   Val F1
---------------------------------------------------------------------------
BiLSTM v3                               0.4116   0.3075   0.5332   0.4104
mBERT                                   0.4655   0.3785   0.5812   0.4854
Ensemble (α=0.45)                       0.4648   0.3859   0.5863   0.4919

Val F1 = performance on Gemini-labeled unfiltered val (for reference)
Gold 2K = performance on human-annotated data (the real test)

>>> Gold 2K scores comparable to val — consistent performance.


## Step 8: Error Analysis by Position

In [28]:
def error_analysis_by_field(gold_samples, pred_key, field, model_name):
    """Break down F1 by a categorical field (position, complexity, etc.)."""
    groups = defaultdict(lambda: {"preds": [], "labels": []})
    for s in gold_samples:
        key = s.get(field, "UNKNOWN")
        groups[key]["preds"].extend(s[pred_key])
        groups[key]["labels"].extend(s["labels_aligned"])

    print(f"\n{model_name} — F1 by {field}:")
    print(f"  {'Category':<20} {'N sents':>8} {'N tokens':>9} {'Macro-F1':>10} {'Ins F1':>8}")
    print("  " + "-" * 58)

    for key in sorted(groups.keys()):
        g = groups[key]
        n_sents = sum(1 for s in gold_samples if s.get(field) == key)
        n_tokens = len(g["labels"])
        mf1 = f1_score(g["labels"], g["preds"], average="macro", zero_division=0)
        if1 = f1_score(g["labels"], g["preds"], labels=[1,2], average="macro", zero_division=0)
        print(f"  {key:<20} {n_sents:>8} {n_tokens:>9,} {mf1:>10.4f} {if1:>8.4f}")

# Run for ensemble (best model)
error_analysis_by_field(gold_samples, "ensemble_preds", "position", "Ensemble")
error_analysis_by_field(gold_samples, "ensemble_preds", "complexity", "Ensemble")
error_analysis_by_field(gold_samples, "ensemble_preds", "participle_type", "Ensemble")


Ensemble — F1 by position:
  Category              N sents  N tokens   Macro-F1   Ins F1
  ----------------------------------------------------------
  UNKNOWN                  2008    44,932     0.4648   0.3858

Ensemble — F1 by complexity:
  Category              N sents  N tokens   Macro-F1   Ins F1
  ----------------------------------------------------------
  unknown                  2008    44,932     0.4648   0.3858

Ensemble — F1 by participle_type:
  Category              N sents  N tokens   Macro-F1   Ins F1
  ----------------------------------------------------------
  unknown                  2008    44,932     0.4648   0.3858


## Step 9: Per-Class Confusion Detail

In [31]:
def show_confusion_detail(all_labels, all_preds, model_name):
    """Show where each model makes mistakes."""
    print(f"\n{model_name} — Error patterns:")
    for true_cls in range(NUM_LABELS):
        for pred_cls in range(NUM_LABELS):
            if true_cls == pred_cls:
                continue
            count = sum(1 for l, p in zip(all_labels, all_preds)
                       if l == true_cls and p == pred_cls)
            if count > 0:
                print(f"  {LABEL_LIST[true_cls]:>15} → {LABEL_LIST[pred_cls]:<15}: {count:>6,}")

show_confusion_detail(ensemble_res["all_labels"], ensemble_res["all_preds"], "Ensemble")
show_confusion_detail(mbert_res["all_labels"], mbert_res["all_preds"], "mBERT")


Ensemble — Error patterns:
                O → COMMA_AFTER    :    337
                O → BUTH_AFTER     :    689
                O → REMOVE_COMMA   :     49
      COMMA_AFTER → O              :     53
      COMMA_AFTER → BUTH_AFTER     :     16
       BUTH_AFTER → O              :     38
       BUTH_AFTER → COMMA_AFTER    :      8
       BUTH_AFTER → REMOVE_COMMA   :      1
     REMOVE_COMMA → O              :     31
     REMOVE_COMMA → COMMA_AFTER    :      5
     REMOVE_COMMA → BUTH_AFTER     :      3

mBERT — Error patterns:
                O → COMMA_AFTER    :    362
                O → BUTH_AFTER     :    732
                O → REMOVE_COMMA   :     34
      COMMA_AFTER → O              :     47
      COMMA_AFTER → BUTH_AFTER     :     16
       BUTH_AFTER → O              :     34
       BUTH_AFTER → COMMA_AFTER    :     10
       BUTH_AFTER → REMOVE_COMMA   :      1
     REMOVE_COMMA → O              :     27
     REMOVE_COMMA → COMMA_AFTER    :      9
     REMOVE_COMMA → BUT

## Step 10: Save Results

In [33]:
results = {"evaluation_set": "gold_2k",
            "n_sentences": len(gold_samples),
            "n_tokens": sum(s["n_aligned"] for s in gold_samples),
            "ensemble_alpha": ENSEMBLE_ALPHA,
            "bilstm": {k: v for k, v in bilstm_res.items() if k not in ("all_preds", "all_labels")},
            "mbert": {k: v for k, v in mbert_res.items() if k not in ("all_preds", "all_labels")},
            "ensemble": {k: v for k, v in ensemble_res.items() if k not in ("all_preds", "all_labels")},
    "comparison_with_val": {"bilstm_val": 0.4104, # Reference values from Gemini-labeled validation set (for comparison)
                            "mbert_val": 0.4854,
                            "ensemble_val": 0.4919}}

results_path = Path("gold_2k_evaluation_results.json")
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {results_path}")

Results saved to gold_2k_evaluation_results.json


In [35]:
# CORRECTION COVERAGE — exact match on non-O positions

def correction_coverage(gold_samples, pred_key, model_name):
    """Of all human corrections, how many did the model predict correctly?"""
    total_gold_corrections = 0
    exact_matches = 0
    model_extra = 0  # model predicted non-O where gold is O

    # Per-class breakdown
    class_gold = Counter()
    class_matched = Counter()

    for s in gold_samples:
        labels = s["labels_aligned"]
        preds = s[pred_key]

        for lbl, pred in zip(labels, preds):
            if lbl != 0:  # gold has a correction here
                total_gold_corrections += 1
                class_gold[lbl] += 1
                if pred == lbl:  # model got it exactly right
                    exact_matches += 1
                    class_matched[lbl] += 1
            elif pred != 0:  # model predicted correction where gold says O
                model_extra += 1

    coverage = exact_matches / total_gold_corrections if total_gold_corrections > 0 else 0

    print(f"\n{'='*60}")
    print(f"{model_name} — Correction Coverage")
    print(f"{'='*60}")
    print(f"  Gold corrections (non-O labels):  {total_gold_corrections}")
    print(f"  Model matched exactly:            {exact_matches}")
    print(f"  Model false insertions (O→non-O): {model_extra}")
    print(f"  Coverage: {exact_matches}/{total_gold_corrections} = {coverage:.1%}")
    print()
    print(f"  Per-class:")
    for cls_id in sorted(class_gold.keys()):
        name = LABEL_LIST[cls_id]
        g = class_gold[cls_id]
        m = class_matched[cls_id]
        print(f"    {name:15s}: {m:>4}/{g:>4} = {m/g:.1%}")
    print(f"\n  Precision on corrections: {exact_matches}/{exact_matches+model_extra}"
          f" = {exact_matches/(exact_matches+model_extra):.1%}"
          f" (of everything the model flagged, how many were real)")

    return {"total_gold": total_gold_corrections,
            "exact_matches": exact_matches,
            "false_insertions": model_extra,
            "coverage": round(coverage, 4)}

bilstm_cov = correction_coverage(gold_samples, "bilstm_preds", "BiLSTM")
mbert_cov = correction_coverage(gold_samples, "mbert_preds", "mBERT")
ensemble_cov = correction_coverage(gold_samples, "ensemble_preds", "Ensemble (α=0.45)")


BiLSTM v3 — Correction Coverage
  Gold corrections (non-O labels):  514
  Model matched exactly:            300
  Model false insertions (O→non-O): 1325
  Coverage: 300/514 = 58.4%

  Per-class:
    COMMA_AFTER    :  140/ 227 = 61.7%
    BUTH_AFTER     :  156/ 243 = 64.2%
    REMOVE_COMMA   :    4/  44 = 9.1%

  Precision on corrections: 300/1625 = 18.5% (of everything the model flagged, how many were real)

mBERT — Correction Coverage
  Gold corrections (non-O labels):  514
  Model matched exactly:            367
  Model false insertions (O→non-O): 1128
  Coverage: 367/514 = 71.4%

  Per-class:
    COMMA_AFTER    :  164/ 227 = 72.2%
    BUTH_AFTER     :  198/ 243 = 81.5%
    REMOVE_COMMA   :    5/  44 = 11.4%

  Precision on corrections: 367/1495 = 24.5% (of everything the model flagged, how many were real)

Ensemble (α=0.45) — Correction Coverage
  Gold corrections (non-O labels):  514
  Model matched exactly:            359
  Model false insertions (O→non-O): 1075
  Coverage: 359/5